# Control case for Demo 2

## Notes:
> - The customized database is removed.
> - The agent failed the task.

#### Bo Ni, March 31, 2026

## 0. Check the floor

In [1]:
# # make into one block
# # 
# {
#     "tags": [
#         "hide-input",
#     ]
# }
# # 
#@CHech the base
import os, sys
print('Here is : \n', os.popen('pwd').read())
print('What we get in hardware: \n', os.popen('nvidia-smi').read())
kernel_name = os.path.basename(sys.executable.replace("/bin/python",""))
print ("The VirEnv kernal in action: ", kernel_name)
# 
# # make it work for both py and notebook
# try: 
#     from jupyter_client import kernelspec
#     spec = kernelspec.get_kernel_spec(kernel_name)
#     print("Path to it: ", spec.resource_dir)
# except:
#     print ("This suppose to be a .py run")
# # /path/to/my/kernel
try: 
    import torch
    print("What we have in software: \n Torch version:", torch.__version__) 
    print('Python: ', sys.version) # no switch case code
    # 
    print('What hardware the software see:')
    device = torch.device(
        "cuda:0" if torch.cuda.is_available() else "cpu"
    )
    print(device)
    num_of_gpus = torch.cuda.device_count()
    print("# of GPU", num_of_gpus)
    # 
    torch.cuda.empty_cache()
except:
    print ("No torch founded")
# 

Here is : 
 /trace/group/tmousavi/bni2/1_JupyterGit/workspace/TC_Python_LLM_1/8_build_demos

What we get in hardware: 
 Tue Mar 31 15:54:00 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 545.23.08              Driver Version: 545.23.08    CUDA Version: 12.3     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A40                     On  | 00000000:25:00.0 Off |                    0 |
|  0%   20C    P8              12W / 300W |      4MiB / 46068MiB |      0%      Default |
|                                     

In [2]:
# global setup
CKeys={}
CKeys['Debug']=1

In [3]:
# for TC-python 
# 
from dotenv import load_dotenv
load_dotenv('./local_safe/env_setup.custom')

True

In [4]:
# test
if CKeys['Debug']==1:
    print ( os.environ.get('LSHOST') )
    print ( os.environ.get('TC24A_HOME') )

172.24.116.37
/trace/group/tmousavi/software/Thermo-Calc/2024a


## Working part

In [5]:
import json
import os

import chromadb

import autogen
from autogen import AssistantAgent
from autogen.agentchat.contrib.retrieve_user_proxy_agent import RetrieveUserProxyAgent

# Accepted file formats for that can be stored in
# a vector database instance
from autogen.retrieve_utils import TEXT_FORMATS

from autogen.coding import LocalCommandLineCodeExecutor

from IPython.display import Image

In [6]:
# config_list = autogen.config_list_from_json("OAI_CONFIG_LIST")

# ref:
# https://github.com/microsoft/autogen/blob/main/autogen/oai/openai_utils.py
config_list = autogen.config_list_from_json(
    env_or_file="Simple_OpenAI_Keys_Safe.json",
    file_location="../Local_Store/",
    filter_dict={
        "model": [
            "gpt-4o-2024-05-13",
            # "gpt-4o-2024-08-06",
            # "gpt-4o-mini",
        ],
    },
)

assert len(config_list) > 0
print("models to use: ", [config_list[i]["model"] for i in range(len(config_list))])

models to use:  ['gpt-4o-2024-05-13']


### Local setup

In [7]:
wk_dir = "../Local_Store/For_8_1_1/"


In [8]:
# clean everything

if CKeys['Debug']==1:
    ## clean everything
    if os.path.exists(wk_dir):
        shutil.rmtree(wk_dir)
        print (f"Remove everything...")

In [9]:
# file store

if not os.path.exists(wk_dir):
    print (f"Create working path...")
    os.mkdir(wk_dir)

Create working path...


In [10]:
# subdir
code_dir = wk_dir + "coding/"

### Construct agents

In [11]:
if CKeys['Debug']==1:
    print("Accepted file formats for `docs_path`:")
    print(TEXT_FORMATS)
    print (config_list[0]["model"])
    # print (config_list[1]["model"])

Accepted file formats for `docs_path`:
['txt', 'json', 'csv', 'tsv', 'md', 'html', 'htm', 'rtf', 'rst', 'jsonl', 'log', 'xml', 'yaml', 'yml', 'pdf', 'mdx']
gpt-4o-2024-05-13


In [12]:
# 1. create an AssistantAgent instance named "assistant"
# 
assistant = AssistantAgent(
    name="Assistant",
    system_message="""
You are a helpful expert in computational mateiral science, especially CALPHAD.
Based on the provided examples and information, you write python script to run Themro-Calc via tc_python to do calculations, like single equlibrium and Scheil calculaitons.
You also write python code for postprocess and debug the code.
Note, all needed packages are already installed. 
You don't repeat the code block.
""",
    llm_config={
        "timeout": 1200,
        "cache_seed": 42,
        "config_list": config_list,
        "temperature": 0.3,
    },
)

In [13]:
# create a UserProxyAgent instance named "user_proxy"

user_proxy = autogen.UserProxyAgent(
    name="User_Proxy",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=10,
    is_termination_msg=lambda x: x.get("content", "").rstrip().endswith("TERMINATE"),
    code_execution_config={
        # the executor to run the generated code
        "executor": LocalCommandLineCodeExecutor(
            work_dir= code_dir
        ),
    },
    system_message="""
    tc_python package is already installed. You never install any package, but just run the code.
"""
)

### Examples

#### Try some problems from the learning hub

In [14]:
# =======================================================================

code_problem_ver_3 = """
Task:
Perform a Scheil-calculation for the given alloy composition and plot the latent heat of solidification:
Fe - 1 wt-% C - 10 wt-% Cr

Please solve the task, print/plot the result 
and save them to json or png files.

"""


In [15]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

In [16]:
# 
chat_result_wo_RAG = user_proxy.initiate_chat(
    assistant, 
    message=code_problem_ver_3,
    # 
    summary_method="reflection_with_llm",
)


User_Proxy (to Assistant):


Task:
Perform a Scheil-calculation for the given alloy composition and plot the latent heat of solidification:
Fe - 1 wt-% C - 10 wt-% Cr

Please solve the task, print/plot the result 
and save them to json or png files.



--------------------------------------------------------------------------------
Assistant (to User_Proxy):

Sure, I can help you with that. Below is the Python script to perform a Scheil calculation for the given alloy composition using Thermo-Calc via the `tc_python` package. The script will also plot the latent heat of solidification and save the results to JSON and PNG files.

```python
import json
import matplotlib.pyplot as plt
import numpy as np
from tc_python import *

# Initialize Thermo-Calc
with TCPython() as session:
    # Define the database and elements
    session.select_database_and_elements(database_name="TCFE9", elements=["Fe", "C", "Cr"])

    # Set the composition
    composition = {"C": 1.0, "Cr": 10.0, "Fe": 89.0}  